### Import statements

In [ ]:
import ccc2026
ccc2026.setup()
from ccc2026 import run_training, plot_training, rolling_samples, plot_rolling, feature_count, all_prefs
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from matplotlib import pyplot as plt

### Initialize state

In [2]:
state = {}

### Utility functions

In [3]:
def bs_alert(msg, cls="warning"):
    '''Display message as a bootstrap alert'''

    html = f'<div class="alert alert-{cls}" role="alert">{msg}</div>'
    return HTML(html)

### Define window view controls

In [4]:
# work dropdown
work_dd = widgets.Dropdown(
    description = "Work",
    options = [work for work in all_prefs],
)

# prefix dropdown
pref_dd = widgets.Dropdown(
    description = "Book",
    options = [pref for pref in all_prefs[work_dd.value]],
)

# figure viewport
roll_view = widgets.Output()

# handler for work change
def on_work_change(change):
    '''on change to work, update pref options'''
    pref_dd.options = [pref for pref in all_prefs[work_dd.value]]

    # trigger pref update
    on_pref_change(None)

# handler for any change to view
def on_pref_change(change):
    '''redraw the rolling window plot'''
    
    if "test" not in state:
        return
    with roll_view:
        clear_output(wait=True)
        fig = plot_rolling(state["test"], work=work_dd.value, pref=pref_dd.value)
        display(fig)
        plt.close(fig)

# register handlers with widgets
work_dd.observe(on_work_change, names="value")
pref_dd.observe(on_pref_change, names="value")

### Define feature selection controls

In [5]:
# dictionary holds per-feature-class feature input
feat_input = {}

# create one empty textarea per feature class
for label in feature_count:
    feat_input[label] = widgets.Textarea(
        description = "Include:",
    )

# define the tabbed container to hold them
tab_container = widgets.Tab()
tab_container.children = [ta for ta in feat_input.values()]
tab_container.titles = [label for label in feat_input]

def reset_features():
    '''(Re)populate each textarea with its default feature list'''

    for label in feat_input:
        feat_input[label].value = "\n".join([feat.strip() for feat in feature_count[label].index.values])

# populate textareas with default values
reset_features()

# reset button
reset_btn = widgets.Button(
    description = "Reset to defaults",
)
reset_btn.on_click(lambda btn: reset_features())

# a function to validate contents and return a feature set dict
def get_valid_features():
    '''Parse the contents of textareas and extract valid features by feature class

    Feature classes with no valid features are omitted entirely, so that
    run_training can skip them.
    '''

    fs = {}

    for label in feat_input:
        features = [feat for feat in feat_input[label].value.split() if feat in feature_count[label].index.values]
        if features:
            fs[label] = features

    return fs

### Define training controls

In [6]:
# sample size
sample_size_input = widgets.BoundedIntText(
    value = 1000,
    min = 200,
    max = 5000,
    description = "Sample size"
)

# random number seed
seed_input = widgets.BoundedIntText(
    value = 0,
    min = 0,
    max = 2**53 - 1,
    description = "Seed"
)

# run button
train_btn = widgets.Button(
    description = "Run training",
)

# model viewport
train_view = widgets.Output()

# minimum number of features required across all classes (PCA below uses 3 components)
MIN_FEATURES = 3

# handler for button click
def on_train_click(btn):
    '''train model, project rolling samples'''

    # for now just use base features
    feature_set = get_valid_features()

    # need enough features (across one or more classes) for the PCA step
    n_features = sum(len(features) for features in feature_set.values())
    if n_features < MIN_FEATURES:
        with train_view:
            clear_output(wait=True)
            display(bs_alert(f"Select at least {MIN_FEATURES} features (across one or more categories) before running training."))
        return

    # run training
    state["train"] = run_training(
        feature_set = feature_set,
        sample_size = sample_size_input.value,
        seed = seed_input.value,
    )

    # display model
    with train_view:
        clear_output(wait=True)
        fig = plot_training(state["train"])
        display(fig)
        plt.close(fig)

    # calculate rolling window and project
    state["test"] = rolling_samples(state["train"])

    # trigger rolling viewport update
    on_pref_change(None)

# register handler for train button
train_btn.on_click(on_train_click)

## Interactive UI

### Feature Selection

In [7]:
display(tab_container)
display(reset_btn)

Button(description='Reset to defaults', style=ButtonStyle())

### Training

In [8]:
display(widgets.HBox([sample_size_input, seed_input, train_btn]))
display(train_view)

Output()

### Rolling Samples

In [9]:
display(widgets.HBox([work_dd, pref_dd]))
display(roll_view)

Output()